# reWatt Colab: YAMNet (audio) + YOLO26N (vision)

This notebook trains:
- YAMNet embedding classifier head for human-present vs empty
- YOLO26N person detector for counting

Assumptions:
- Audio clips are 5s at 16 kHz mono
- Vision dataset is a single class: person

In [ ]:
# Install dependencies
import sys
!{sys.executable} -m pip -q install tensorflow==2.15.0 tensorflow-hub==0.16.1 librosa==0.10.1 ultralytics==8.3.0

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/rewatt_ml'
os.makedirs(ROOT, exist_ok=True)

## Drive layout
Expected folders:
- audio/clips_5s/positive
- audio/clips_5s/negative
- vision/raw_images/train|val|test
- vision/labels_yolo/train|val|test
- exports/yamnet_tflite
- exports/yolo_tflite

In [ ]:
os.makedirs(f'{ROOT}/audio/clips_5s/positive', exist_ok=True)
os.makedirs(f'{ROOT}/audio/clips_5s/negative', exist_ok=True)
os.makedirs(f'{ROOT}/vision/raw_images/train', exist_ok=True)
os.makedirs(f'{ROOT}/vision/raw_images/val', exist_ok=True)
os.makedirs(f'{ROOT}/vision/raw_images/test', exist_ok=True)
os.makedirs(f'{ROOT}/vision/labels_yolo/train', exist_ok=True)
os.makedirs(f'{ROOT}/vision/labels_yolo/val', exist_ok=True)
os.makedirs(f'{ROOT}/vision/labels_yolo/test', exist_ok=True)
os.makedirs(f'{ROOT}/exports/yamnet_tflite', exist_ok=True)
os.makedirs(f'{ROOT}/exports/yolo_tflite', exist_ok=True)

## Audio pipeline (YAMNet)
We train a small classifier head on top of YAMNet embeddings.
Positive class: human-present. Negative class: empty-room.

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np
import librosa

YAMNET_MODEL_HANDLE = 'https://tfhub.dev/google/yamnet/1'
yamnet = hub.load(YAMNET_MODEL_HANDLE)

SAMPLE_RATE = 16000
CLIP_SECONDS = 5

POSITIVE_INDICES = [
    0, 2, 3, 12, 13, 15, 18, 23,
    61, 62, 63, 64, 65,
    36, 42, 43, 44, 45,
    47, 48, 373, 454, 481,
    348, 351, 352, 353, 354, 355,
    378, 380, 381, 473, 485
]

In [ ]:
def load_audio_5s(path):
    audio, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True, duration=CLIP_SECONDS)
    if len(audio) < SAMPLE_RATE * CLIP_SECONDS:
        pad = SAMPLE_RATE * CLIP_SECONDS - len(audio)
        audio = np.pad(audio, (0, pad))
    return audio.astype(np.float32)

def yamnet_embeddings(path):
    audio = load_audio_5s(path)
    scores, embeddings, _ = yamnet(audio)
    emb = tf.reduce_mean(embeddings, axis=0)
    score = tf.reduce_mean(scores, axis=0)
    return emb.numpy(), score.numpy()

In [ ]:
import glob
from sklearn.model_selection import train_test_split

POS_DIR = f'{ROOT}/audio/clips_5s/positive'
NEG_DIR = f'{ROOT}/audio/clips_5s/negative'

pos_files = glob.glob(os.path.join(POS_DIR, '*.wav'))
neg_files = glob.glob(os.path.join(NEG_DIR, '*.wav'))

X = []
y = []

for f in pos_files:
    emb, _ = yamnet_embeddings(f)
    X.append(emb)
    y.append(1)

for f in neg_files:
    emb, _ = yamnet_embeddings(f)
    X.append(emb)
    y.append(0)

X = np.array(X)
y = np.array(y)

X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=42)

In [ ]:
from tensorflow import keras

model = keras.Sequential([
    keras.layers.Input(shape=(X.shape[1],)),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc')]
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=32
)

model.evaluate(X_test, y_test)

In [ ]:
def representative_data_gen():
    for i in range(min(200, len(X_train))):
        yield [X_train[i:i+1].astype(np.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

yamnet_head_tflite = converter.convert()
yamnet_out = f'{ROOT}/exports/yamnet_tflite/yamnet_head_int8.tflite'
with open(yamnet_out, 'wb') as f:
    f.write(yamnet_head_tflite)

print('Saved:', yamnet_out)

## Vision pipeline (YOLO26N)
Single class dataset: person (class id 0).

In [ ]:
data_yaml = f'''path: {ROOT}/vision
train: raw_images/train
val: raw_images/val
test: raw_images/test

names:
  0: person
'''
with open(f'{ROOT}/vision/data.yaml', 'w') as f:
    f.write(data_yaml)
print('Wrote:', f'{ROOT}/vision/data.yaml')

In [ ]:
from ultralytics import YOLO

# Replace yolo26n.pt with your base or pretrained weights if needed
yolo = YOLO('yolo26n.pt')

yolo.train(
    data=f'{ROOT}/vision/data.yaml',
    imgsz=640,
    epochs=100,
    batch=16,
    device=0,
,
,
,

: 
,
: {
: 

: [
,
,
,
,
,
,
,
,
,

: 
,
: {
: 

: [
,
  
: 
,
: {
: 

: [

,


,

: 
,
: {
: 

: [
,